# Week 4 — PQC Baseline Comparison

本 Notebook 延續 W2 / W3 的 **PQC Retrieval + Ranking** 題目。使用 12 筆 Scenario / 18 筆 Evidence。

**提醒：** W3 的 weak label 尚未人工確認，這裡的 Recall@5 是「與 weak label 比對的診斷值」，不是正式測試成績；本週也沒有正式 Train/Validation/Test Group Split。

In [ ]:
from pathlib import Path
import csv
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from week04_pqc_baselines import run, load_csv

scenarios = load_csv("pqc_scenario_dataset_v0.1.csv")
evidence = load_csv("pqc_evidence_corpus_v0.1.csv")
print("Scenario:", len(scenarios), "Evidence:", len(evidence))
print("Device platform groups:", sorted({s["device_platform"] for s in scenarios}))
print("Label status:", sorted({s["label_status"] for s in scenarios}))


## 1. Fair comparison

三個方法在相同 Scenario、Evidence、CPU、RAM、Flash 和安全需求條件下執行。弱標籤只能拿來**評估參考吻合度**，**不能當成預測輸入**。

- Naive Function-only：只依密碼功能找參數。
- Top-1 Secure：符合硬性條件後，只留 operation cycles 最少的參數。
- Rule-based Filtering：符合硬性條件後，保留最多 5 個參數。

為避免把 benchmark 的 stack 值誤解成完整記憶體需求，資源檢查僅作為基本 evidence-based screening。

In [ ]:
results, failures = run()
for item in results:
    print(f"{item['method']}: weak-label recall@5={item['weak_label_recall_at_5_diagnostic']:.4f}, "
          f"security violation={item['security_violation_count']}/{item['candidate_count']}, "
          f"function mismatch={item['function_mismatch_count']}")


## 2. Failure cases

前兩筆是安全等級硬條件違反；第三筆是相對 weak labels 的候選遺漏，不應寫成已人工確認的 False Negative。

In [ ]:
for f in failures:
    print("\n", f["case_id"], f["method"])
    print("Prediction:", f["predicted_candidates"])
    print("Failure:", f["failure_type"])
    print("Status:", f["evidence_status"])


## 3. Baseline 依據與限制

- W2 有 Manual Search + Rule-based Filtering；**Manual Search 尚未真正量測，不填假數據**。
- 目前兩個硬體 group，不改成 random split；正式 Group Split 須等平台增加。
- 三個方法的執行時間只代表程式運算成本，**不能對照人工搜尋時間**。
- 需要人工確認 relevant candidate set，才能產出正式的 Recall@5。

輸出 CSV 位於 `../outputs/`；提交內容詳見 `../evidence/week04_baseline_report.md`。

In [ ]:
assert len(scenarios) == 12
assert len(evidence) == 18
assert all(s["label_status"] == "needs_human_review" for s in scenarios)
assert len(failures) == 3
rb = next(r for r in results if r["method"] == "Rule-based Filtering")
assert rb["security_violation_count"] == 0
assert rb["function_mismatch_count"] == 0
print("Basic validation passed; formal Recall@5 still N/A.")
